# ONNX Runtime 推理教程 (ONNX Runtime Inference Tutorial)

> **前置知识**: PyTorch 基础、ONNX 模型格式
>
> **学习目标**: 掌握 ONNX Runtime 推理引擎的使用和优化

---

## 什么是 ONNX Runtime？

```
┌─────────────────────────────────────────────────────────────┐
│                    ONNX Runtime 架构                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  训练框架                    ONNX Runtime                   │
│  ┌─────────────┐            ┌─────────────────────────┐    │
│  │  PyTorch    │            │                         │    │
│  │  TensorFlow │ ──ONNX──→  │  Execution Providers    │    │
│  │  Keras      │            │  ┌─────────────────┐    │    │
│  └─────────────┘            │  │ CPU (默认)      │    │    │
│                             │  │ CUDA (GPU)      │    │    │
│                             │  │ TensorRT        │    │    │
│                             │  │ CoreML (Apple)  │    │    │
│                             │  │ OpenVINO (Intel)│    │    │
│                             │  └─────────────────┘    │    │
│                             └─────────────────────────┘    │
│                                                             │
│  核心优势:                                                  │
│  - 跨平台: Windows/Linux/macOS/iOS/Android                 │
│  - 跨框架: 统一的 ONNX 格式                                │
│  - 高性能: 图优化、算子融合、内存复用                      │
│  - 易集成: Python/C++/C#/Java 多语言支持                   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **ONNX 模型导出** - PyTorch → ONNX
2. **推理会话创建** - InferenceSession 配置
3. **执行提供者** - CPU/GPU 加速
4. **图优化级别** - 性能优化选项
5. **性能基准测试** - 延迟和吞吐量测量

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import tempfile
import os
import time

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# 检查 ONNX Runtime 是否可用
try:
    import onnxruntime as ort
    ORT_AVAILABLE = True
    print("=" * 60)
    print("环境准备完成")
    print("=" * 60)
    print(f"PyTorch 版本: {torch.__version__}")
    print(f"ONNX Runtime 版本: {ort.__version__}")
    print(f"\n可用的执行提供者 (Execution Providers):")
    for provider in ort.get_available_providers():
        print(f"  - {provider}")
except ImportError:
    ORT_AVAILABLE = False
    print("ONNX Runtime 未安装")
    print("安装命令: pip install onnxruntime  # CPU 版本")
    print("         pip install onnxruntime-gpu  # GPU 版本")

## 1. 创建测试模型并导出为 ONNX

**核心概念**: ONNX (Open Neural Network Exchange) 是跨框架的模型交换格式

```
┌─────────────────────────────────────────────────────────────┐
│                    ONNX 导出流程                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  PyTorch Model                                              │
│       │                                                     │
│       ▼                                                     │
│  torch.onnx.export()                                        │
│       │                                                     │
│       ├── 追踪模型执行路径                                  │
│       ├── 转换为 ONNX 算子                                  │
│       └── 序列化为 .onnx 文件                               │
│       │                                                     │
│       ▼                                                     │
│  ONNX Model (.onnx)                                         │
│       │                                                     │
│       ├── input_names: 输入节点名称                         │
│       ├── output_names: 输出节点名称                        │
│       ├── dynamic_axes: 动态维度 (如可变 batch)             │
│       └── opset_version: ONNX 算子集版本                    │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 定义测试模型
# ============================================================

class ImageClassifier(nn.Module):
    """
    图像分类模型
    
    一个简单的 CNN 模型，用于演示 ONNX Runtime 推理
    
    结构:
    - 2 个卷积层 (带 BatchNorm)
    - 2 个全连接层
    - Dropout 正则化
    """
    def __init__(self, num_classes=10):
        super().__init__()
        # 卷积层 1: 3 通道 → 32 通道
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        # 卷积层 2: 32 通道 → 64 通道
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        # 池化层
        self.pool = nn.MaxPool2d(2)
        
        # 全连接层
        self.fc1 = nn.Linear(64 * 8 * 8, 256)  # 64通道 × 8×8 特征图
        self.fc2 = nn.Linear(256, num_classes)
        
        # Dropout
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        # 卷积块 1: Conv → BN → ReLU → Pool
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # [B,3,32,32] → [B,32,16,16]
        
        # 卷积块 2: Conv → BN → ReLU → Pool
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # [B,32,16,16] → [B,64,8,8]
        
        # 展平
        x = x.view(x.size(0), -1)  # [B,64,8,8] → [B,4096]
        
        # 全连接层
        x = self.dropout(F.relu(self.fc1(x)))  # [B,4096] → [B,256]
        return self.fc2(x)  # [B,256] → [B,10]


# 创建模型并设置为评估模式
model = ImageClassifier()
model.eval()  # 重要: 导出前必须设置为 eval 模式!

# 统计参数
total_params = sum(p.numel() for p in model.parameters())

print("=" * 60)
print("测试模型信息")
print("=" * 60)
print(f"\n模型结构:")
print(f"  输入: [B, 3, 32, 32]")
print(f"  Conv1: 3→32 通道, 3x3 卷积")
print(f"  Conv2: 32→64 通道, 3x3 卷积")
print(f"  FC1: 4096→256")
print(f"  FC2: 256→10")
print(f"\n总参数量: {total_params:,}")

In [ ]:
# ============================================================
# 导出为 ONNX 格式
# ============================================================

# 创建示例输入
dummy_input = torch.randn(1, 3, 32, 32)

# 创建临时目录保存模型
model_dir = tempfile.mkdtemp()
onnx_path = os.path.join(model_dir, "model.onnx")

# 导出 ONNX 模型
torch.onnx.export(
    model,                          # PyTorch 模型
    dummy_input,                    # 示例输入 (用于追踪)
    onnx_path,                      # 输出路径
    input_names=['input'],          # 输入节点名称
    output_names=['output'],        # 输出节点名称
    dynamic_axes={                  # 动态维度配置
        'input': {0: 'batch_size'},   # batch 维度可变
        'output': {0: 'batch_size'}
    },
    opset_version=14,               # ONNX 算子集版本
    do_constant_folding=True        # 常量折叠优化
)

print("=" * 60)
print("ONNX 导出完成")
print("=" * 60)
print(f"\n保存路径: {onnx_path}")
print(f"模型大小: {os.path.getsize(onnx_path) / (1024*1024):.2f} MB")

# 验证 ONNX 模型 (可选)
try:
    import onnx
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print(f"ONNX 模型验证: ✓ 通过")
except ImportError:
    print("提示: 安装 onnx 包可进行模型验证 (pip install onnx)")

## 2. 创建推理会话 (InferenceSession)

**核心概念**: InferenceSession 是 ONNX Runtime 的核心类，负责加载模型和执行推理

```
┌─────────────────────────────────────────────────────────────┐
│                    InferenceSession 工作流程                 │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 加载 ONNX 模型                                          │
│     session = ort.InferenceSession("model.onnx")           │
│                                                             │
│  2. 获取输入/输出信息                                       │
│     inputs = session.get_inputs()                          │
│     outputs = session.get_outputs()                        │
│                                                             │
│  3. 执行推理                                                │
│     result = session.run(output_names, input_dict)         │
│                                                             │
│  关键参数:                                                  │
│  - providers: 执行提供者列表 (CPU/CUDA/TensorRT...)        │
│  - sess_options: 会话配置选项                              │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 创建基本推理会话
# ============================================================
if ORT_AVAILABLE:
    # 创建推理会话
    session = ort.InferenceSession(onnx_path)
    
    print("=" * 60)
    print("推理会话信息")
    print("=" * 60)
    
    # 查看输入信息
    print("\n输入节点:")
    for inp in session.get_inputs():
        print(f"  名称: {inp.name}")
        print(f"  形状: {inp.shape}")
        print(f"  类型: {inp.type}")
    
    # 查看输出信息
    print("\n输出节点:")
    for out in session.get_outputs():
        print(f"  名称: {out.name}")
        print(f"  形状: {out.shape}")
        print(f"  类型: {out.type}")
    
    # 查看使用的执行提供者
    print(f"\n使用的执行提供者: {session.get_providers()}")

In [ ]:
# ============================================================
# 执行推理
# ============================================================
if ORT_AVAILABLE:
    # 准备输入数据 (必须是 numpy 数组，float32 类型)
    input_data = np.random.randn(1, 3, 32, 32).astype(np.float32)
    
    # 方法 1: 使用字典传入输入
    # session.run(output_names, input_dict)
    # - output_names: None 表示返回所有输出
    # - input_dict: {输入名称: 输入数据}
    outputs = session.run(None, {'input': input_data})
    
    print("=" * 60)
    print("推理结果")
    print("=" * 60)
    print(f"\n输入形状: {input_data.shape}")
    print(f"输出形状: {outputs[0].shape}")
    print(f"输出示例 (前5个): {outputs[0][0][:5].round(4)}")

In [ ]:
# ============================================================
# 验证 ONNX Runtime 与 PyTorch 输出一致性
# ============================================================
if ORT_AVAILABLE:
    # 使用相同输入比较 PyTorch 和 ONNX Runtime 的输出
    with torch.no_grad():
        torch_input = torch.from_numpy(input_data)
        torch_output = model(torch_input).numpy()
    
    # 计算差异
    diff = np.abs(outputs[0] - torch_output).max()
    
    print("=" * 60)
    print("输出一致性验证")
    print("=" * 60)
    print(f"\nPyTorch 输出 (前5个): {torch_output[0][:5].round(4)}")
    print(f"ONNX RT 输出 (前5个): {outputs[0][0][:5].round(4)}")
    print(f"\n最大差异: {diff:.8f}")
    print(f"输出一致: {'✓ 通过' if diff < 1e-5 else '✗ 失败'}")

## 3. 会话配置选项 (SessionOptions)

**核心概念**: SessionOptions 用于配置推理会话的优化级别、线程数、内存策略等

```
┌─────────────────────────────────────────────────────────────┐
│                    SessionOptions 配置项                     │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  图优化级别 (graph_optimization_level):                     │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  ORT_DISABLE_ALL    - 禁用所有优化                  │   │
│  │  ORT_ENABLE_BASIC   - 基本优化 (常量折叠)           │   │
│  │  ORT_ENABLE_EXTENDED - 扩展优化 (算子融合)          │   │
│  │  ORT_ENABLE_ALL     - 所有优化 (推荐)               │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  线程配置:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  intra_op_num_threads - 算子内并行线程数            │   │
│  │  inter_op_num_threads - 算子间并行线程数            │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  执行模式:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  ORT_SEQUENTIAL - 顺序执行 (低延迟)                 │   │
│  │  ORT_PARALLEL   - 并行执行 (高吞吐)                 │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 配置会话选项
# ============================================================
if ORT_AVAILABLE:
    # 创建会话选项
    sess_options = ort.SessionOptions()
    
    # ============================================================
    # 图优化级别
    # ============================================================
    # ORT_DISABLE_ALL: 禁用所有优化
    # ORT_ENABLE_BASIC: 基本优化 (常量折叠等)
    # ORT_ENABLE_EXTENDED: 扩展优化 (算子融合等)
    # ORT_ENABLE_ALL: 所有优化 (推荐)
    sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    
    # ============================================================
    # 线程配置
    # ============================================================
    sess_options.intra_op_num_threads = 4  # 算子内并行线程数
    sess_options.inter_op_num_threads = 2  # 算子间并行线程数
    
    # ============================================================
    # 执行模式
    # ============================================================
    # ORT_SEQUENTIAL: 顺序执行 (低延迟，适合单请求)
    # ORT_PARALLEL: 并行执行 (高吞吐，适合批处理)
    sess_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    
    # ============================================================
    # 内存优化
    # ============================================================
    sess_options.enable_mem_pattern = True   # 启用内存模式优化
    sess_options.enable_mem_reuse = True     # 启用内存复用
    
    # 创建优化后的会话
    optimized_session = ort.InferenceSession(
        onnx_path,
        sess_options=sess_options,
        providers=['CPUExecutionProvider']
    )
    
    print("=" * 60)
    print("优化会话配置")
    print("=" * 60)
    print(f"\n图优化级别: ORT_ENABLE_ALL")
    print(f"算子内线程数: 4")
    print(f"算子间线程数: 2")
    print(f"执行模式: SEQUENTIAL")
    print(f"内存优化: 启用")
    print(f"\n使用的提供者: {optimized_session.get_providers()}")

In [ ]:
# ============================================================
# 保存优化后的模型
# ============================================================
if ORT_AVAILABLE:
    # 可以将优化后的模型保存到文件
    optimized_path = os.path.join(model_dir, "model_optimized.onnx")
    
    sess_options_save = ort.SessionOptions()
    sess_options_save.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess_options_save.optimized_model_filepath = optimized_path  # 指定保存路径
    
    # 创建会话时会自动保存优化后的模型
    _ = ort.InferenceSession(onnx_path, sess_options=sess_options_save)
    
    print("=" * 60)
    print("优化模型保存")
    print("=" * 60)
    print(f"\n原始模型大小: {os.path.getsize(onnx_path) / 1024:.2f} KB")
    print(f"优化模型大小: {os.path.getsize(optimized_path) / 1024:.2f} KB")
    print(f"\n优化后的模型已保存到: {optimized_path}")

## 4. 执行提供者 (Execution Providers)

**核心概念**: Execution Provider (EP) 是 ONNX Runtime 的硬件抽象层，支持不同硬件加速

```
┌─────────────────────────────────────────────────────────────┐
│                    Execution Providers                       │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Provider              硬件           安装包                │
│  ─────────────────────────────────────────────────────────  │
│  CPUExecutionProvider  CPU (默认)     onnxruntime           │
│  CUDAExecutionProvider NVIDIA GPU     onnxruntime-gpu       │
│  TensorrtExecutionProvider TensorRT   onnxruntime-gpu       │
│  ROCMExecutionProvider AMD GPU        onnxruntime-rocm      │
│  DmlExecutionProvider  DirectML       onnxruntime-directml  │
│  CoreMLExecutionProvider Apple        onnxruntime           │
│  OpenVINOExecutionProvider Intel      onnxruntime-openvino  │
│                                                             │
│  优先级机制:                                                │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  providers=['CUDAExecutionProvider',                │   │
│  │             'CPUExecutionProvider']                 │   │
│  │                                                     │   │
│  │  → 优先使用 CUDA，不可用时回退到 CPU                │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 查看可用的执行提供者
# ============================================================
if ORT_AVAILABLE:
    available_providers = ort.get_available_providers()
    
    # 常见执行提供者说明
    provider_info = {
        'CPUExecutionProvider': ('CPU (默认)', 'onnxruntime'),
        'CUDAExecutionProvider': ('NVIDIA GPU', 'onnxruntime-gpu'),
        'TensorrtExecutionProvider': ('NVIDIA TensorRT', 'onnxruntime-gpu'),
        'ROCMExecutionProvider': ('AMD GPU', 'onnxruntime-rocm'),
        'DmlExecutionProvider': ('DirectML (Windows)', 'onnxruntime-directml'),
        'CoreMLExecutionProvider': ('Apple CoreML', 'onnxruntime'),
        'OpenVINOExecutionProvider': ('Intel OpenVINO', 'onnxruntime-openvino'),
    }
    
    print("=" * 60)
    print("执行提供者 (Execution Providers)")
    print("=" * 60)
    print(f"\n{'Provider':<30} {'硬件':<20} {'状态':<10}")
    print("-" * 60)
    
    for provider, (desc, package) in provider_info.items():
        status = "✓ 可用" if provider in available_providers else "✗ 未安装"
        print(f"{provider:<30} {desc:<20} {status:<10}")

In [ ]:
# ============================================================
# 使用特定的执行提供者创建会话
# ============================================================
if ORT_AVAILABLE:
    def create_session_with_provider(model_path, providers):
        """
        创建使用指定提供者的会话
        
        参数:
            model_path: ONNX 模型路径
            providers: 执行提供者列表 (按优先级排序)
            
        返回:
            session: 推理会话 (如果创建失败返回 None)
        """
        try:
            session = ort.InferenceSession(model_path, providers=providers)
            actual_providers = session.get_providers()
            print(f"请求提供者: {providers}")
            print(f"实际使用: {actual_providers}")
            return session
        except Exception as e:
            print(f"创建失败: {e}")
            return None
    
    print("=" * 60)
    print("创建不同提供者的会话")
    print("=" * 60)
    
    # CPU 会话
    print("\n--- CPU 会话 ---")
    cpu_session = create_session_with_provider(onnx_path, ['CPUExecutionProvider'])
    
    # 尝试 CUDA 会话 (如果可用)
    if 'CUDAExecutionProvider' in available_providers:
        print("\n--- CUDA 会话 ---")
        cuda_session = create_session_with_provider(
            onnx_path, 
            ['CUDAExecutionProvider', 'CPUExecutionProvider']
        )

## 5. 性能基准测试

**核心概念**: 通过基准测试评估推理性能，包括延迟、吞吐量等指标

```
┌─────────────────────────────────────────────────────────────┐
│                    性能指标说明                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  延迟 (Latency):                                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  - Mean: 平均延迟                                   │   │
│  │  - P50: 50% 请求的延迟 (中位数)                     │   │
│  │  - P90: 90% 请求的延迟                              │   │
│  │  - P99: 99% 请求的延迟 (尾延迟)                     │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  吞吐量 (Throughput):                                       │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  - samples/sec: 每秒处理的样本数                    │   │
│  │  - 吞吐量 = batch_size / latency                    │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  测试方法:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 预热 (Warmup): 运行几次让系统稳定               │   │
│  │  2. 多次运行: 收集统计数据                          │   │
│  │  3. 计算统计量: 均值、标准差、百分位数              │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 性能基准测试函数
# ============================================================
if ORT_AVAILABLE:
    def benchmark_session(session, input_data, num_runs=100, warmup=10):
        """
        基准测试推理性能
        
        参数:
            session: ONNX Runtime 推理会话
            input_data: 输入数据 (numpy 数组)
            num_runs: 测试运行次数
            warmup: 预热运行次数
            
        返回:
            dict: 包含各种性能指标
        """
        input_name = session.get_inputs()[0].name
        
        # 预热 (让系统缓存稳定)
        for _ in range(warmup):
            session.run(None, {input_name: input_data})
        
        # 计时测试
        latencies = []
        for _ in range(num_runs):
            start = time.perf_counter()
            session.run(None, {input_name: input_data})
            latencies.append((time.perf_counter() - start) * 1000)  # 转换为毫秒
        
        latencies = np.array(latencies)
        
        return {
            'mean_ms': np.mean(latencies),
            'std_ms': np.std(latencies),
            'min_ms': np.min(latencies),
            'max_ms': np.max(latencies),
            'p50_ms': np.percentile(latencies, 50),
            'p90_ms': np.percentile(latencies, 90),
            'p99_ms': np.percentile(latencies, 99),
            'throughput': 1000 / np.mean(latencies)  # samples/sec
        }
    
    # 测试不同批次大小
    batch_sizes = [1, 4, 8, 16, 32]
    results = []
    
    print("=" * 60)
    print("性能基准测试")
    print("=" * 60)
    print(f"\n测试配置: {len(batch_sizes)} 种批次大小, 每种 50 次运行")
    print("\n正在测试...")
    
    for batch_size in batch_sizes:
        input_data = np.random.randn(batch_size, 3, 32, 32).astype(np.float32)
        stats = benchmark_session(optimized_session, input_data, num_runs=50)
        stats['batch_size'] = batch_size
        stats['total_throughput'] = stats['throughput'] * batch_size
        results.append(stats)
    
    # 显示结果
    print(f"\n{'Batch':<8} {'Mean(ms)':<12} {'P50(ms)':<12} {'P99(ms)':<12} {'吞吐量(samples/s)':<20}")
    print("-" * 64)
    for r in results:
        print(f"{r['batch_size']:<8} {r['mean_ms']:<12.3f} {r['p50_ms']:<12.3f} {r['p99_ms']:<12.3f} {r['total_throughput']:<20.1f}")

In [ ]:
# ============================================================
# 可视化性能结果
# ============================================================
if ORT_AVAILABLE:
    try:
        import matplotlib.pyplot as plt
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        # 图1: 延迟 vs 批次大小
        axes[0].bar([str(r['batch_size']) for r in results], 
                    [r['mean_ms'] for r in results], 
                    color='steelblue', alpha=0.7, edgecolor='black')
        axes[0].set_xlabel('Batch Size')
        axes[0].set_ylabel('Latency (ms)')
        axes[0].set_title('推理延迟 vs 批次大小')
        axes[0].grid(axis='y', alpha=0.3)
        
        # 图2: 吞吐量 vs 批次大小
        axes[1].bar([str(r['batch_size']) for r in results], 
                    [r['total_throughput'] for r in results], 
                    color='coral', alpha=0.7, edgecolor='black')
        axes[1].set_xlabel('Batch Size')
        axes[1].set_ylabel('Throughput (samples/sec)')
        axes[1].set_title('吞吐量 vs 批次大小')
        axes[1].grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print("\n观察:")
        print("  - 延迟随批次大小增加而增加 (但非线性)")
        print("  - 吞吐量随批次大小增加而增加 (批处理效率)")
        print("  - 选择批次大小需要权衡延迟和吞吐量")
    except ImportError:
        print("matplotlib 未安装，跳过可视化")

## 6. 图优化级别对比

**核心概念**: 不同优化级别对性能的影响

```
┌─────────────────────────────────────────────────────────────┐
│                    图优化级别对比                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  级别                优化内容                  适用场景     │
│  ─────────────────────────────────────────────────────────  │
│  DISABLE_ALL        无优化                    调试         │
│  ENABLE_BASIC       常量折叠、死代码消除      快速部署     │
│  ENABLE_EXTENDED    + 算子融合                生产环境     │
│  ENABLE_ALL         + 所有高级优化            最佳性能     │
│                                                             │
│  常见优化:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  常量折叠: 预计算常量表达式                         │   │
│  │  死代码消除: 移除不影响输出的计算                   │   │
│  │  算子融合: Conv+BN+ReLU → 单一算子                  │   │
│  │  内存优化: 张量复用、原地操作                       │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 测试不同优化级别的性能
# ============================================================
if ORT_AVAILABLE:
    optimization_levels = [
        (ort.GraphOptimizationLevel.ORT_DISABLE_ALL, "禁用优化"),
        (ort.GraphOptimizationLevel.ORT_ENABLE_BASIC, "基本优化"),
        (ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED, "扩展优化"),
        (ort.GraphOptimizationLevel.ORT_ENABLE_ALL, "全部优化"),
    ]
    
    input_data = np.random.randn(8, 3, 32, 32).astype(np.float32)
    opt_results = []
    
    print("=" * 60)
    print("图优化级别性能对比")
    print("=" * 60)
    print("\n正在测试不同优化级别...")
    
    for level, name in optimization_levels:
        opts = ort.SessionOptions()
        opts.graph_optimization_level = level
        
        sess = ort.InferenceSession(onnx_path, sess_options=opts)
        
        # 预热
        for _ in range(10):
            sess.run(None, {'input': input_data})
        
        # 计时
        times = []
        for _ in range(50):
            start = time.perf_counter()
            sess.run(None, {'input': input_data})
            times.append((time.perf_counter() - start) * 1000)
        
        opt_results.append({
            'name': name,
            'mean_ms': np.mean(times),
            'std_ms': np.std(times)
        })
    
    # 显示结果
    print(f"\n{'优化级别':<15} {'平均延迟(ms)':<15} {'标准差(ms)':<15} {'加速比':<10}")
    print("-" * 55)
    baseline = opt_results[0]['mean_ms']
    for r in opt_results:
        speedup = baseline / r['mean_ms']
        print(f"{r['name']:<15} {r['mean_ms']:<15.3f} {r['std_ms']:<15.3f} {speedup:<10.2f}x")

In [ ]:
<cell_type>markdown</cell_type>## 7. 动态输入形状测试

**核心概念**: ONNX 模型可以支持动态维度，允许在推理时使用不同的输入形状

```
┌─────────────────────────────────────────────────────────────┐
│                    动态维度配置                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  导出时配置:                                                │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  dynamic_axes = {                                   │   │
│  │      'input': {0: 'batch_size'},  # 第0维动态      │   │
│  │      'output': {0: 'batch_size'}                    │   │
│  │  }                                                  │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  推理时:                                                    │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  batch=1:  [1, 3, 32, 32] → [1, 10]                │   │
│  │  batch=5:  [5, 3, 32, 32] → [5, 10]                │   │
│  │  batch=20: [20, 3, 32, 32] → [20, 10]              │   │
│  │                                                     │   │
│  │  同一个模型，不同批次大小!                          │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

# ============================================================
# 测试动态批次大小
# ============================================================
if ORT_AVAILABLE:
    print("=" * 60)
    print("动态输入形状测试")
    print("=" * 60)
    
    # 测试不同批次大小
    test_batch_sizes = [1, 5, 10, 20, 50]
    
    print(f"\n使用同一个模型测试不同批次大小:")
    print(f"{'Batch Size':<15} {'输入形状':<25} {'输出形状':<20}")
    print("-" * 60)
    
    for batch_size in test_batch_sizes:
        # 创建不同批次大小的输入
        input_data = np.random.randn(batch_size, 3, 32, 32).astype(np.float32)
        
        # 执行推理
        output = optimized_session.run(None, {'input': input_data})
        
        print(f"{batch_size:<15} {str(input_data.shape):<25} {str(output[0].shape):<20}")
    
    print(f"\n✓ 动态批次大小工作正常!")
    print(f"  同一个 ONNX 模型可以处理任意批次大小的输入")

In [ ]:
<cell_type>markdown</cell_type>## 8. IO Binding (高级优化)

**核心概念**: IO Binding 允许预分配输入/输出缓冲区，减少数据拷贝开销

```
┌─────────────────────────────────────────────────────────────┐
│                    IO Binding 优化                           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  传统方式 (每次推理都有数据拷贝):                           │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  NumPy → 拷贝到设备 → 推理 → 拷贝回 NumPy           │   │
│  │         ↑ 开销大                ↑ 开销大            │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  IO Binding (预分配缓冲区):                                 │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  预分配设备缓冲区 → 直接推理 → 结果已在缓冲区       │   │
│  │  ↑ 一次性开销        ↑ 无拷贝开销                   │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  适用场景:                                                  │
│  - 高频推理 (每秒数百次)                                   │
│  - GPU 推理 (减少 CPU-GPU 数据传输)                        │
│  - 流式处理 (连续输入)                                     │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

# ============================================================
# IO Binding 示例 (CPU 版本)
# ============================================================
if ORT_AVAILABLE:
    print("=" * 60)
    print("IO Binding 示例")
    print("=" * 60)
    
    # 创建 IO Binding 对象
    io_binding = optimized_session.io_binding()
    
    # 准备输入数据
    input_data = np.random.randn(8, 3, 32, 32).astype(np.float32)
    
    # 创建 OrtValue (ONNX Runtime 的张量对象)
    input_ortvalue = ort.OrtValue.ortvalue_from_numpy(input_data, 'cpu', 0)
    
    # 绑定输入
    io_binding.bind_ortvalue_input('input', input_ortvalue)
    
    # 绑定输出 (让 ONNX Runtime 自动分配)
    io_binding.bind_output('output', 'cpu')
    
    # 执行推理
    optimized_session.run_with_iobinding(io_binding)
    
    # 获取输出
    output_ortvalue = io_binding.get_outputs()[0]
    output_data = output_ortvalue.numpy()
    
    print(f"\n输入形状: {input_data.shape}")
    print(f"输出形状: {output_data.shape}")
    print(f"\n✓ IO Binding 推理成功!")
    print(f"\n提示: IO Binding 在 GPU 推理时效果更明显")
    print(f"      可以避免 CPU-GPU 之间的数据拷贝")

In [ ]:
# ============================================================
# IO Binding 性能对比
# ============================================================
if ORT_AVAILABLE:
    print("=" * 60)
    print("IO Binding vs 标准推理 性能对比")
    print("=" * 60)
    
    input_data = np.random.randn(16, 3, 32, 32).astype(np.float32)
    num_runs = 100
    
    # 标准推理计时
    for _ in range(10):  # 预热
        optimized_session.run(None, {'input': input_data})
    
    standard_times = []
    for _ in range(num_runs):
        start = time.perf_counter()
        optimized_session.run(None, {'input': input_data})
        standard_times.append((time.perf_counter() - start) * 1000)
    
    # IO Binding 推理计时
    io_binding = optimized_session.io_binding()
    input_ortvalue = ort.OrtValue.ortvalue_from_numpy(input_data, 'cpu', 0)
    io_binding.bind_ortvalue_input('input', input_ortvalue)
    io_binding.bind_output('output', 'cpu')
    
    for _ in range(10):  # 预热
        optimized_session.run_with_iobinding(io_binding)
    
    iobinding_times = []
    for _ in range(num_runs):
        start = time.perf_counter()
        optimized_session.run_with_iobinding(io_binding)
        iobinding_times.append((time.perf_counter() - start) * 1000)
    
    print(f"\n{'方法':<20} {'平均延迟(ms)':<15} {'标准差(ms)':<15}")
    print("-" * 50)
    print(f"{'标准推理':<20} {np.mean(standard_times):<15.3f} {np.std(standard_times):<15.3f}")
    print(f"{'IO Binding':<20} {np.mean(iobinding_times):<15.3f} {np.std(iobinding_times):<15.3f}")
    
    speedup = np.mean(standard_times) / np.mean(iobinding_times)
    print(f"\nIO Binding 加速比: {speedup:.2f}x")
    print(f"\n注意: CPU 上 IO Binding 加速效果有限")
    print(f"      GPU 上效果更明显 (减少 CPU-GPU 数据传输)")

<cell_type>markdown</cell_type>## 总结

本教程介绍了 ONNX Runtime 的核心功能和最佳实践：

### 核心知识点

| 主题 | 关键内容 |
|:-----|:---------|
| 模型导出 | PyTorch → ONNX，动态维度配置 |
| 推理会话 | InferenceSession 创建和配置 |
| 执行提供者 | CPU/CUDA/TensorRT 等硬件加速 |
| 图优化 | DISABLE_ALL → ENABLE_ALL 四个级别 |
| 性能测试 | 延迟、吞吐量、百分位数指标 |
| IO Binding | 预分配缓冲区，减少数据拷贝 |

### ONNX Runtime API 速查

```python
import onnxruntime as ort

# 基本推理
session = ort.InferenceSession("model.onnx")
output = session.run(None, {'input': input_data})

# 指定执行提供者
session = ort.InferenceSession(
    "model.onnx",
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)

# 配置会话选项
opts = ort.SessionOptions()
opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
opts.intra_op_num_threads = 4
session = ort.InferenceSession("model.onnx", sess_options=opts)

# IO Binding (高级)
io_binding = session.io_binding()
io_binding.bind_ortvalue_input('input', input_ortvalue)
io_binding.bind_output('output', 'cpu')
session.run_with_iobinding(io_binding)
```

### 最佳实践

```
性能优化检查清单:
✓ 使用 ORT_ENABLE_ALL 优化级别
✓ 根据硬件选择合适的执行提供者
✓ 配置合适的线程数 (intra_op/inter_op)
✓ 使用动态批次大小提高灵活性
✓ 高频推理场景使用 IO Binding
✓ 通过基准测试找到最优配置

常见问题:
✗ 忘记设置 model.eval() 导致输出不一致
✗ 输入数据类型不是 float32
✗ 未使用图优化导致性能不佳
```

### 下一步学习

- **02_TensorRT_tutorial.ipynb**: TensorRT 极致 GPU 优化
- **03_vLLM_tutorial.ipynb**: LLM 专用推理引擎
- **04_Advanced_Inference_tutorial.ipynb**: 高级推理技术

In [ ]:
# ============================================================
# 清理临时文件
# ============================================================
import shutil

# 清理临时目录
shutil.rmtree(model_dir, ignore_errors=True)

print("=" * 60)
print("清理完成")
print("=" * 60)
print("\n✓ 临时文件已清理")
print("\n本教程演示了 ONNX Runtime 的核心功能:")
print("  1. ONNX 模型导出")
print("  2. InferenceSession 创建和配置")
print("  3. 执行提供者 (Execution Providers)")
print("  4. 图优化级别")
print("  5. 性能基准测试")
print("  6. 动态输入形状")
print("  7. IO Binding 高级优化")